In [1]:
library('haven')
library('dplyr')
library('data.table')
#install.packages("ukbnmr")
#library('ukbnmr')
source('NMR_DataPrepFunctions.R')


Attaching package: ‘dplyr’


The following objects are masked from ‘package:stats’:

    filter, lag


The following objects are masked from ‘package:base’:

    intersect, setdiff, setequal, union



Attaching package: ‘data.table’


The following objects are masked from ‘package:dplyr’:

    between, first, last




In [2]:
#nmr_raw = fread('../Data/Nightingale/Nightingale_Raw/csv/ukb674581.csv')
#processed <- remove_technical_variation(nmr_raw)  


In [3]:
nmr = fread('../Data/Nightingale/NMR_NotDerived.csv')
nmr_composite = nightingale_composite_biomarker_compute(nmr)
nmr_1p = apply_log_transformation(as.data.frame(nmr_composite), 'eid')
nmr_ratio = nightingale_ratio_compute(nmr)
nmr_ext = extended_ratios_compute(nmr_ratio)
nmr_ext <- select(nmr_ext, c(eid, setdiff(names(nmr_ext), names(nmr_1p))))
nmr_df = full_join(nmr_1p, nmr_ext)

Joining with `by = join_by(eid)`


In [4]:
fwrite(nmr_df, 'nmr_with_composite_ratio_extend.csv')

In [5]:
summary(nmr_df)

      eid          Clinical_LDL_C    VLDL_size        LDL_size    
 Min.   :1000030   Min.   :0.000   Min.   :3.541   Min.   :3.198  
 1st Qu.:2260836   1st Qu.:1.116   1st Qu.:3.656   1st Qu.:3.214  
 Median :3514580   Median :1.259   Median :3.679   Median :3.216  
 Mean   :3514137   Mean   :1.247   Mean   :3.679   Mean   :3.216  
 3rd Qu.:4771733   3rd Qu.:1.391   3rd Qu.:3.702   3rd Qu.:3.219  
 Max.   :6024869   Max.   :1.952   Max.   :3.796   Max.   :3.229  
                                                                  
    HDL_size      Phosphoglyc        Cholines      Phosphatidylc  
 Min.   :2.310   Min.   :0.5231   Min.   :0.6603   Min.   :0.457  
 1st Qu.:2.350   1st Qu.:1.1039   1st Qu.:1.1913   1st Qu.:1.046  
 Median :2.361   Median :1.1836   Median :1.2675   Median :1.126  
 Mean   :2.364   Mean   :1.1819   Mean   :1.2654   Mean   :1.124  
 3rd Qu.:2.376   3rd Qu.:1.2621   3rd Qu.:1.3422   3rd Qu.:1.204  
 Max.   :2.445   Max.   :1.6487   Max.   :1.7178   Max.   :1.5

In [6]:
metabolomics_aging_score_coef = fread('Metabolomic_aging_score_coef.csv', data.table = FALSE)
summary(metabolomics_aging_score_coef$Coefficient)

      Min.    1st Qu.     Median       Mean    3rd Qu.       Max. 
-0.1993884 -0.0389986 -0.0008316 -0.0062906  0.0279512  0.2220550 

In [7]:
coef_vector <- setNames(
  metabolomics_aging_score_coef$Coefficient,
  metabolomics_aging_score_coef$`Aging-related representative biomarker`
)
biomarkers <- names(coef_vector)
nmr_subset <- as.data.frame(select(nmr_df, c(eid, all_of(biomarkers))))

nmr_subset[, biomarkers] <- lapply(nmr_subset[, biomarkers], function(col) {
  col[is.infinite(col)] <- NA
  return(col)
})

nmr_subset$metabolomics_aging_score <- rowSums(sweep(nmr_subset[, biomarkers], 2, coef_vector, `*`), na.rm=T)


In [8]:
inf_check <- sapply(nmr_subset[, biomarkers], function(col) any(is.infinite(col)))
which(inf_check)

named integer(0)

In [9]:
summary(nmr_subset$metabolomics_aging_score)

     Min.   1st Qu.    Median      Mean   3rd Qu.      Max. 
-5950.667    -8.788    -8.115    -8.284    -7.334     1.489 